# Migrating from Document AI to AI_EXTRACT

This demo shows how to migrate from the deprecated **Document AI** (`model!PREDICT`) to the new **AI_EXTRACT** function.

### Why Migrate?
- Document AI and `model!PREDICT` are **deprecated** (BCR-2156)
- AI_EXTRACT is **zero-shot** — no model training required
- Powered by **Arctic-extract** vision model
- Part of the broader **AISQL** function family

### Migration Comparison
| Document AI (Old) | AI_EXTRACT (New) |
|-------------------|------------------|
| Model creation in Snowsight UI | No setup needed |
| Fine-tuning required for accuracy | Zero-shot, ready to use |
| `model!PREDICT(GET_PRESIGNED_URL(...))` | `AI_EXTRACT(file => TO_FILE(...), ...)` |
| Arctic-TILT model | Arctic-extract vision model |
| Confidence scores | Flexible response formats |

## Setup: Create Database, Schema, and Stage

In [ ]:
USE DATABASE DOC_AI_DEPRECATION;
USE SCHEMA PUBLIC;

In [ ]:
CREATE OR REPLACE STAGE DEPRECATED_DOC_AI_IMAGES
    DIRECTORY = (ENABLE = TRUE AUTO_REFRESH = TRUE)
    ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE');

CREATE OR REPLACE STAGE DEMO_DOCS
    DIRECTORY = (ENABLE = TRUE AUTO_REFRESH = TRUE)
    ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE');

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

MY_STAGE = 'DEMO_DOCS'
MY_FILE_NAME = 'data/*'

put_result = session.file.put(MY_FILE_NAME, MY_STAGE, auto_compress=False, overwrite=True)

In [ ]:
ALTER STAGE DOC_AI_DEPRECATION.PUBLIC.DEMO_DOCS REFRESH;
SELECT * FROM DIRECTORY(@DOC_AI_DEPRECATION.PUBLIC.DEMO_DOCS);

---
## The Old Way: Document AI with model!PREDICT

**Problems with the old approach:**
- Required creating and training a model in Snowsight UI
- Fine-tuning needed for good accuracy
- Model management overhead
- Limited flexibility in question formats

In [ ]:
SELECT 
    DOC_AI_DEPRECATION.PUBLIC.DEPRECATION_DEMO!PREDICT(
        GET_PRESIGNED_URL(@DEMO_DOCS, relative_path),
        1
    ) AS predictions
FROM DIRECTORY(@DEMO_DOCS)
LIMIT 1;

---
## Migration Path: Preserving Your Fine-Tuned Model

If you have a fine-tuned Document AI model you want to preserve, follow these steps:

1. Step 1: Go to AI Studio -> Document Processing Playground -> Go To Document AI Builds -> Migrate Now
2. Step 2: Select a Model and Export to a Stage (If you want to access priorly trained on documents with their prompts and responses)
3. Step 3: Convert Current Pipelines to use AI_EXTRACT() instead of MODEL!PREDICT()

In [ ]:
CREATE OR REPLACE FILE FORMAT my_json
  TYPE = 'JSON';

In [ ]:
CREATE OR REPLACE TABLE exported_data_table AS (
   SELECT
      input_file.$1:file AS file,
      input_file.$1:prompt AS prompt,
      input_file.$1:annotatedResponse AS response
   FROM '@DOC_AI_DEPRECATION.PUBLIC.DEPRECATED_DOC_AI_IMAGES/DOC_AI_DEPRECATION_PUBLIC_DEPRECATION_DEMO_2026_01_20_19_36_40/annotations.jsonl' (FILE_FORMAT => my_json) input_file
   WHERE response != '{}'
);

In [ ]:
SELECT
    *
FROM EXPORTED_DATA_TABLE;

In [ ]:
SELECT AI_EXTRACT(
    MODEL => 'DOC_AI_DEPRECATION.PUBLIC.DEPRECATION_DEMO',
    FILE => TO_FILE('@DOC_AI_DEPRECATION.PUBLIC.DEMO_DOCS', 'Manual_2022-02-01.pdf')
) AS result;

---
## The New Way: AI_EXTRACT

AI_EXTRACT is a **zero-shot** function — no model training required. Just ask questions!

## Document 1: Simple Gene Table (CpG Sites)

A straightforward 4-column table with gene names, numeric values, and text descriptions.

In [ ]:
from PIL import Image
import os

img_path = os.path.join(os.getcwd(), "data", "simple_table_data.jpg")
img = Image.open(img_path)
img

In [ ]:
SELECT AI_EXTRACT(
    file => TO_FILE('@DOC_AI_DEPRECATION.PUBLIC.DEMO_DOCS', 'simple_table_data.jpg'),
    responseFormat => {
        'schema': {
            'type': 'object',
            'properties': {
                'gene_insight_table': {
                    'description': 'CpG Sites and genes of interest(Genes with significant TSS200',
                    'type': 'object',
                    'column_ordering': ['gene', 'case', 'control', 'ipa_network'],
                    'properties': {
                        'gene': {'type': 'array'},
                        'case': {'type': 'array'},
                        'control': {'type': 'array'},
                        'ipa_network': {'type': 'array'}
                    }
                }
            }
        }
    }
) AS extracted_genes;

In [ ]:
import json
result = json.loads(cells.cell10.to_pandas()['EXTRACTED_GENES'][0])
print(json.dumps(result, indent=2))

In [ ]:
CREATE TABLE IF NOT EXISTS prompt_templates (
    template_id VARCHAR PRIMARY KEY,
    response_format VARIANT
);

In [ ]:
INSERT INTO PROMPT_TEMPLATES
VALUES
('GENE_TABLE',
PARSE_JSON("{
        'schema': {
            'type': 'object',
            'properties': {
                'gene_insight_table': {
                    'description': 'CpG Sites and genes of interest(Genes with significant TSS200',
                    'type': 'object',
                    'column_ordering': ['gene', 'case', 'control', 'ipa_network'],
                    'properties': {
                        'gene': {'type': 'array'},
                        'case': {'type': 'array'},
                        'control': {'type': 'array'},
                        'ipa_network': {'type': 'array'}
                    }
                }
            }
        }
    }'"));

## Document 2: Dual-Column Scientific Table (Gene Sequence Identity)

A table with gene identifiers and multiple numeric columns showing identity percentages and KaKs values.

In [ ]:
img_path = os.path.join(os.getcwd(), "data", "dual_column_table_data.jpg")
img = Image.open(img_path)
img

In [ ]:
CREATE TEMPORARY TABLE DOC_AI_DEPRECATION.PUBLIC.DUAL_COLUMN_TABLE as
SELECT AI_EXTRACT(
    file => TO_FILE('@DOC_AI_DEPRECATION.PUBLIC.DEMO_DOCS', 'dual_column_table_data.jpg'),
    responseFormat => {
        'schema': {
            'type': 'object',
            'properties': {
                'sequence_id_table': {
                    'description': 'Interspecific sequence identities and Ka/Ks values for seven genes',
                    'type': 'object',
                    'column_ordering': ['gene_identifier', 'identity_rice_maize', 'identity_rice_sorghum', 'identity_sorghum_maize', 'kaks_rice_maize', 'kaks_rice_sorghum', 'kaks_sorghum_maize', 'size'],
                    'properties': {
                        'gene_identifier': {'type': 'array'},
                        'identity_rice_maize': {'type': 'array'},
                        'identity_rice_sorghum': {'type': 'array'},
                        'identity_sorghum_maize': {'type': 'array'},
                        'kaks_rice_maize': {'type': 'array'},
                        'kaks_rice_sorghum': {'type': 'array'},
                        'kaks_sorghum_maize': {'type': 'array'},
                        'size': {'type': 'array'}
                    }
                }
            }
        }
    }
) AS extracted_sequence_ids;

In [ ]:
SELECT 
      f.index AS row_num,
      f.value::STRING AS gene_identifier,
      t.extracted_sequence_ids:response:sequence_id_table:identity_rice_maize[f.index]::STRING AS identity_rice_maize,
      t.extracted_sequence_ids:response:sequence_id_table:identity_rice_sorghum[f.index]::STRING AS identity_rice_sorghum,
      t.extracted_sequence_ids:response:sequence_id_table:identity_sorghum_maize[f.index]::STRING AS identity_sorghum_maize,
      t.extracted_sequence_ids:response:sequence_id_table:kaks_rice_maize[f.index]::STRING AS kaks_rice_maize,
      t.extracted_sequence_ids:response:sequence_id_table:kaks_rice_sorghum[f.index]::STRING AS kaks_rice_sorghum,
      t.extracted_sequence_ids:response:sequence_id_table:kaks_sorghum_maize[f.index]::STRING AS kaks_sorghum_maize,
      t.extracted_sequence_ids:response:sequence_id_table:size[f.index]::STRING AS size
  FROM DOC_AI_DEPRECATION.PUBLIC.DUAL_COLUMN_TABLE t,
  LATERAL FLATTEN(input => t.extracted_sequence_ids:response:sequence_id_table:gene_identifier) f

## Document 3: Multi-Layer Nested Table (Medical Demographics)

A complex table with hierarchical categories (Age group, FIGO, Morphology, Surgery, Radiotherapy) and multiple population columns.

In [ ]:
img_path = os.path.join(os.getcwd(), "data", "multi_layer_nested_table_data.jpg")
img = Image.open(img_path)
img

In [ ]:
CREATE TEMPORARY TABLE DOC_AI_DEPRECATION.PUBLIC.MULTI_LAYER_NESTED_TABLE_DATA_FLATTENED as
SELECT AI_EXTRACT(
      TO_FILE('@DOC_AI_DEPRECATION.PUBLIC.DEMO_DOCS', 'multi_layer_nested_table_data.jpg'),
      {
        'schema': {
            'type': 'object',
            'properties': {
                'tumor_characteristics': {
                  'type': 'object',
                  'column_ordering': ['category', 'subcategory', 'philippine_freq', 'philippine_pct', 'filipino_american_freq', 'filipino_american_pct', 'caucasian_freq', 'caucasian_pct', 'p_value'],
                  'properties': {
                      'category': {'type': 'array', 'description': 'Main category: Age group, FIGO, Morphology, Surgery, or Radiotherapy'},
                      'subcategory': {'type': 'array', 'description': 'Subcategory value within the main category'},
                      'philippine_freq': {'type': 'array'},
                      'philippine_pct': {'type': 'array'},
                      'filipino_american_freq': {'type': 'array'},
                      'filipino_american_pct': {'type': 'array'},
                      'caucasian_freq': {'type': 'array'},
                      'caucasian_pct': {'type': 'array'},
                      'p_value': {'type': 'array', 'description': 'p-value applies to entire category group'}
                  }
              }        
           }
        }
      }
  ) AS extracted_tumor_data;

In [ ]:
SELECT 
      f.index AS row_num,
      f.value::STRING AS category,
      t.extracted_tumor_data:response:tumor_characteristics:subcategory[f.index]::STRING AS subcategory,
      t.extracted_tumor_data:response:tumor_characteristics:philippine_freq[f.index]::INTEGER AS philippine_freq,
      t.extracted_tumor_data:response:tumor_characteristics:philippine_pct[f.index]::STRING AS philippine_pct,
      t.extracted_tumor_data:response:tumor_characteristics:filipino_american_freq[f.index]::INTEGER AS filipino_american_freq,
      t.extracted_tumor_data:response:tumor_characteristics:filipino_american_pct[f.index]::STRING AS filipino_american_pct,
      t.extracted_tumor_data:response:tumor_characteristics:caucasian_freq[f.index]::INTEGER AS caucasian_freq,
      t.extracted_tumor_data:response:tumor_characteristics:caucasian_pct[f.index]::STRING AS caucasian_pct,
      t.extracted_tumor_data:response:tumor_characteristics:p_value[f.index]::STRING AS p_value
  FROM DOC_AI_DEPRECATION.PUBLIC.MULTI_LAYER_NESTED_TABLE_DATA_FLATTENED t,
  LATERAL FLATTEN(input => t.extracted_tumor_data:response:tumor_characteristics:category) f

In [ ]:
CREATE TEMPORARY TABLE DOC_AI_DEPRECATION.PUBLIC.MULTI_LAYER_NESTED_TABLE_DATA_SEPARATE as
SELECT AI_EXTRACT(
      TO_FILE('@DOC_AI_DEPRECATION.PUBLIC.DEMO_DOCS', 'multi_layer_nested_table_data.jpg'),
      {
        'schema': {
            'type': 'object',
            'properties': {
                'age_group': {
                  'type': 'object',
                  'column_ordering': ['subcategory', 'philippine_freq', 'philippine_pct', 'filipino_american_freq', 'filipino_american_pct', 'caucasian_freq', 'caucasian_pct'],
                  'properties': {
                      'subcategory': {'type': 'array', 'description': 'Age ranges: < 40, 40-49, 50-59, 60-69, 70+'},
                      'philippine_freq': {'type': 'array'},
                      'philippine_pct': {'type': 'array'},
                      'filipino_american_freq': {'type': 'array'},
                      'filipino_american_pct': {'type': 'array'},
                      'caucasian_freq': {'type': 'array'},
                      'caucasian_pct': {'type': 'array'}
                  }
              },
              'figo': {
                  'type': 'object',
                  'column_ordering': ['subcategory', 'philippine_freq', 'philippine_pct', 'filipino_american_freq', 'filipino_american_pct', 'caucasian_freq', 'caucasian_pct'],
                  'properties': {
                      'subcategory': {'type': 'array', 'description': 'FIGO stages: I, II, III, IV, Unknown'},
                      'philippine_freq': {'type': 'array'},
                      'philippine_pct': {'type': 'array'},
                      'filipino_american_freq': {'type': 'array'},
                      'filipino_american_pct': {'type': 'array'},
                      'caucasian_freq': {'type': 'array'},
                      'caucasian_pct': {'type': 'array'}
                  }
              },
              'morphology': {
                  'type': 'object',
                  'column_ordering': ['subcategory', 'philippine_freq', 'philippine_pct', 'filipino_american_freq', 'filipino_american_pct', 'caucasian_freq', 'caucasian_pct'],
                  'properties': {
                      'subcategory': {'type': 'array', 'description': 'Morphology types: Serous, Clear cell, Endometrioid, Mucinous, Others, NOS'},
                      'philippine_freq': {'type': 'array'},
                      'philippine_pct': {'type': 'array'},
                      'filipino_american_freq': {'type': 'array'},
                      'filipino_american_pct': {'type': 'array'},
                      'caucasian_freq': {'type': 'array'},
                      'caucasian_pct': {'type': 'array'}
                  }
              },
              'surgery': {
                  'type': 'object',
                  'column_ordering': ['subcategory', 'philippine_freq', 'philippine_pct', 'filipino_american_freq', 'filipino_american_pct', 'caucasian_freq', 'caucasian_pct'],
                  'properties': {
                      'subcategory': {'type': 'array', 'description': 'Surgery status: With surgery, Without surgery, Unknown'},
                      'philippine_freq': {'type': 'array'},
                      'philippine_pct': {'type': 'array'},
                      'filipino_american_freq': {'type': 'array'},
                      'filipino_american_pct': {'type': 'array'},
                      'caucasian_freq': {'type': 'array'},
                      'caucasian_pct': {'type': 'array'}
                  }
              },
              'radiotherapy': {
                  'type': 'object',
                  'column_ordering': ['subcategory', 'philippine_freq', 'philippine_pct', 'filipino_american_freq', 'filipino_american_pct', 'caucasian_freq', 'caucasian_pct'],
                  'properties': {
                      'subcategory': {'type': 'array', 'description': 'Radiotherapy status: With radiotherapy, Without radiotherapy, Unknown'},
                      'philippine_freq': {'type': 'array'},
                      'philippine_pct': {'type': 'array'},
                      'filipino_american_freq': {'type': 'array'},
                      'filipino_american_pct': {'type': 'array'},
                      'caucasian_freq': {'type': 'array'},
                      'caucasian_pct': {'type': 'array'}
                  }
              },
              'p_values': {
                  'type': 'object',
                  'properties': {
                      'age_group': {'type': 'array'},
                      'figo': {'type': 'array'},
                      'morphology': {'type': 'array'},
                      'surgery': {'type': 'array'},
                      'radiotherapy': {'type': 'array'}
                  }
              }
            }
        }
      }
  ) AS extracted_tumor_data;

In [ ]:
SELECT 
      'Age group' AS category,
      f.value::STRING AS subcategory,
      t.extracted_tumor_data:response:age_group:philippine_freq[f.index]::INTEGER AS philippine_freq,
      t.extracted_tumor_data:response:age_group:philippine_pct[f.index]::STRING AS philippine_pct,
      t.extracted_tumor_data:response:age_group:filipino_american_freq[f.index]::INTEGER AS filipino_american_freq,
      t.extracted_tumor_data:response:age_group:filipino_american_pct[f.index]::STRING AS filipino_american_pct,
      t.extracted_tumor_data:response:age_group:caucasian_freq[f.index]::INTEGER AS caucasian_freq,
      t.extracted_tumor_data:response:age_group:caucasian_pct[f.index]::STRING AS caucasian_pct
  FROM DOC_AI_DEPRECATION.PUBLIC.MULTI_LAYER_NESTED_TABLE_DATA_SEPARATE t,
  LATERAL FLATTEN(input => t.extracted_tumor_data:response:age_group:subcategory) f

In [ ]:
SELECT 
      'FIGO' AS category,
      f.value::STRING AS subcategory,
      t.extracted_tumor_data:response:figo:philippine_freq[f.index]::INTEGER AS philippine_freq,
      t.extracted_tumor_data:response:figo:philippine_pct[f.index]::STRING AS philippine_pct,
      t.extracted_tumor_data:response:figo:filipino_american_freq[f.index]::INTEGER AS filipino_american_freq,
      t.extracted_tumor_data:response:figo:filipino_american_pct[f.index]::STRING AS filipino_american_pct,
      t.extracted_tumor_data:response:figo:caucasian_freq[f.index]::INTEGER AS caucasian_freq,
      t.extracted_tumor_data:response:figo:caucasian_pct[f.index]::STRING AS caucasian_pct
  FROM DOC_AI_DEPRECATION.PUBLIC.MULTI_LAYER_NESTED_TABLE_DATA_SEPARATE t,
  LATERAL FLATTEN(input => t.extracted_tumor_data:response:figo:subcategory) f

In [ ]:
import os
from PIL import Image

## Document 4: Hand-Written Form

A form with multiple hand-written responses

In [ ]:
from pdf2image import convert_from_path

images = convert_from_path("data/rental_application_example.pdf")
img = images[0]  # First page as PIL Image
img

In [ ]:
WITH extracted AS (
    SELECT AI_EXTRACT(
      file => TO_FILE('@DOC_AI_DEPRECATION.PUBLIC.DEMO_DOCS', 'rental_application_example.pdf'),
      responseFormat => [
        ['applicant_name', 'What is the full name of the applicant?'],
        ['home_phone', 'What is the home phone number?'],
        ['application_date', 'What is the application date?'],
        ['application_number', 'What is the application number?'],
        ['present_address', 'What is the current street address?'],
        ['city', 'What city does the applicant live in?'],
        ['state', 'What state does the applicant live in?'],
        ['zip_code', 'What is the ZIP code?'],
        ['occupancy_from', 'What is the start date of current occupancy?'],
        ['occupancy_to', 'What is the end date of current occupancy?'],
        ['automobile', 'What is the automobile make/year/registration state and number?'],
        ['social_security_number', 'What is the social security number?'],
        ['present_landlord', 'What is the name of the present landlord?'],
        ['present_landlord_phone', 'What is the present landlord phone number?'],
        ['former_landlord', 'What is the name of the former landlord?'],
        ['former_landlord_address', 'What is the former landlord complete address?'],
        ['former_landlord_phone', 'What is the former landlord phone number?'],
        ['current_employer', 'What is the name of the current employer?'],
        ['current_employer_address', 'What is the current employer address?'],
        ['salary', 'What is the salary amount?'],
        ['length_of_employment', 'How long has the applicant been employed?'],
        ['occupation', 'What is the occupation or source of income?'],
        ['type_of_business', 'What is the type of business?'],
        ['former_employer', 'What is the name of the former employer?'],
        ['former_employer_phone', 'What is the former employer phone number?'],
        ['personal_reference_name', 'What is the personal reference name?'],
        ['personal_reference_phone', 'What is the personal reference phone number?'],
        ['credit_reference', 'What is the credit reference?'],
        ['bank_checking', 'What bank is the checking account with?'],
        ['bank_savings', 'What bank is the savings account with?'],
        ['co_tenants', 'What are the names of all co-tenants?'],
        ['apartment_number', 'What is the apartment number or type?'],
        ['total_occupants', 'What is the total number of occupants?'],
        ['num_adults', 'How many adults?'],
        ['num_pets', 'How many pets?'],
        ['minor_children', 'What are the names and ages of minor children?'],
        ['occupancy_date', 'What is the occupancy date?'],
        ['rent_begins', 'What date does rent begin?'],
        ['lease_term_months', 'What is the term of lease in months?'],
        ['lease_from', 'What is the lease start date?'],
        ['lease_to', 'What is the lease end date?'],
        ['convicted_felon', 'Is the applicant a convicted felon (Y/N)?']
      ]
    ) AS rental_application
  )
  SELECT
    rental_application:response:applicant_name::STRING AS applicant_name,
    rental_application:response:home_phone::STRING AS home_phone,
    rental_application:response:application_date::STRING AS application_date,
    rental_application:response:application_number::STRING AS application_number,
    rental_application:response:present_address::STRING AS present_address,
    rental_application:response:city::STRING AS city,
    rental_application:response:state::STRING AS state,
    rental_application:response:zip_code::STRING AS zip_code,
    rental_application:response:occupancy_from::STRING AS occupancy_from,
    rental_application:response:occupancy_to::STRING AS occupancy_to,
    rental_application:response:automobile::STRING AS automobile,
    rental_application:response:social_security_number::STRING AS social_security_number,
    rental_application:response:present_landlord::STRING AS present_landlord,
    rental_application:response:present_landlord_phone::STRING AS present_landlord_phone,
    rental_application:response:former_landlord::STRING AS former_landlord,
    rental_application:response:former_landlord_address::STRING AS former_landlord_address,
    rental_application:response:former_landlord_phone::STRING AS former_landlord_phone,
    rental_application:response:current_employer::STRING AS current_employer,
    rental_application:response:current_employer_address::STRING AS current_employer_address,
    rental_application:response:salary::STRING AS salary,
    rental_application:response:length_of_employment::STRING AS length_of_employment,
    rental_application:response:occupation::STRING AS occupation,
    rental_application:response:type_of_business::STRING AS type_of_business,
    rental_application:response:former_employer::STRING AS former_employer,
    rental_application:response:former_employer_phone::STRING AS former_employer_phone,
    rental_application:response:personal_reference_name::STRING AS personal_reference_name,
    rental_application:response:personal_reference_phone::STRING AS personal_reference_phone,
    rental_application:response:credit_reference::STRING AS credit_reference,
    rental_application:response:bank_checking::STRING AS bank_checking,
    rental_application:response:bank_savings::STRING AS bank_savings,
    rental_application:response:co_tenants::STRING AS co_tenants,
    rental_application:response:apartment_number::STRING AS apartment_number,
    rental_application:response:total_occupants::NUMBER AS total_occupants,
    rental_application:response:num_adults::NUMBER AS num_adults,
    rental_application:response:num_pets::NUMBER AS num_pets,
    rental_application:response:minor_children::STRING AS minor_children,
    rental_application:response:occupancy_date::STRING AS occupancy_date,
    rental_application:response:rent_begins::STRING AS rent_begins,
    rental_application:response:lease_term_months::NUMBER AS lease_term_months,
    rental_application:response:lease_from::STRING AS lease_from,
    rental_application:response:lease_to::STRING AS lease_to,
    rental_application:response:convicted_felon::STRING AS convicted_felon
  FROM extracted;

## Full Document Processing Pipeline

Step by step walkthrough of putting this pipeline from start to finalized data, based off of another Snowflake walk through found at this [link](https://www.snowflake.com/en/developers/guides/create-a-document-processing-pipeline-with-ai-extract/)

In [ ]:
-- STEP 1: Create and Fill Prompt Management Table

CREATE TABLE IF NOT EXISTS prompt_templates (
    template_id VARCHAR PRIMARY KEY,
    response_format VARIANT
);

INSERT INTO prompt_templates VALUES
('INSPECTION_REVIEWS',
    PARSE_JSON('{
        "schema": {
            "type": "object",
            "properties": {
                "list_of_units": {
                    "description": "Extract the table showing all units and their reported conditions",
                    "type": "object",
                    "column_ordering": ["unit_name", "condition"],
                    "properties": {
                        "unit_name": {
                            "description": "Name of the unit",
                            "type": "array"
                        },
                        "condition": {
                            "description": "Condition reported for the unit",
                            "type": "array"
                        }
                    }
                },
                "inspection_date": {
                    "description": "What is the inspection date?",
                    "type": "string"
                },
                "inspection_grade": {
                    "description": "What is the grade?",
                    "type": "string"
                },
                "inspector": {
                    "description": "Who performed the inspection?",
                    "type": "string"
                }
            }
        }
    }'));

-- STEP 2: AI_EXTRACT Wrapper Function

CREATE OR REPLACE FUNCTION extract_document_data(
    stage_name STRING,
    file_path STRING,
    template_id STRING
)
RETURNS VARIANT
LANGUAGE SQL
AS
$$
    SELECT AI_EXTRACT(
        file => TO_FILE(stage_name, file_path),
        responseFormat => (
            SELECT response_format 
            FROM prompt_templates
            WHERE template_id = template_id
        )
    ):response
$$;

-- STEP 3: Create Table to Store Reviews

CREATE TABLE IF NOT EXISTS pdf_reviews (
    file_name VARCHAR,
    file_size VARIANT,
    last_modified VARCHAR,
    snowflake_file_url VARCHAR,
    json_content VARIANT
);

-- STEP 4: Create Stream on Document Stage

CREATE STREAM IF NOT EXISTS my_pdf_stream ON STAGE demo_docs;
ALTER STAGE demo_docs REFRESH;

-- STEP 5: Create Task on Stream for Processing

CREATE OR REPLACE TASK load_new_file_data
    WAREHOUSE = COMPUTE_WH
    SCHEDULE = '1 minutes'
    COMMENT = 'Process new files in the stage and insert data into the pdf_reviews table.'
WHEN SYSTEM$STREAM_HAS_DATA('my_pdf_stream')
AS
INSERT INTO pdf_reviews (
    SELECT
        RELATIVE_PATH AS file_name,
        size AS file_size,
        last_modified,
        file_url AS snowflake_file_url,
        extract_document_data('@demo_docs', RELATIVE_PATH) AS json_content
    FROM my_pdf_stream
    WHERE METADATA$ACTION = 'INSERT'
);

-- Resume the task
ALTER TASK load_new_file_data RESUME;

-- STEP 6: Create Analysis View

CREATE OR REPLACE VIEW pdf_reviews_view AS
SELECT 
    file_name,
    file_size,
    last_modified,
    snowflake_file_url,
    json_content:inspection_date::STRING AS inspection_date,
    json_content:inspection_grade::STRING AS inspection_grade,
    json_content:inspector::STRING AS inspector,
    json_content:list_of_units:unit_name::ARRAY AS list_of_units_name,
    json_content:list_of_units:condition::ARRAY AS list_of_units_condition
FROM pdf_reviews;

-- STEP 7: Create Flattened View (1 Row per Unit)

CREATE OR REPLACE VIEW pdf_reviews_flattened AS
SELECT 
    file_name,
    file_size,
    last_modified,
    snowflake_file_url,
    json_content:inspection_date::STRING AS inspection_date,
    json_content:inspection_grade::STRING AS inspection_grade,
    json_content:inspector::STRING AS inspector,
    f.index AS unit_index,
    f.value::STRING AS unit_name,
    json_content:list_of_units:condition[f.index]::STRING AS unit_condition
FROM pdf_reviews,
LATERAL FLATTEN(input => json_content:list_of_units:unit_name) f;